> 在线讲义：[第 7 章 · 功耗、散热和端云分工](https://wikieden.github.io/robot-edge-inference/lessons/07-power-and-cloud.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 7 章 · 功耗、散热和端云分工

配套讲义：仓库里 `ai-infra-edge/notes/lessons/07-power-and-cloud.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（123.97 ms 来自 GR00T 2 白皮书，13 ms 来自 π0 论文；电池、电机、网络参数是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 一次推理的能量 | $120 \times 0.12397$ | `energy_j(120, 0.12397)` | 14.9 J |
| 第一步 只算计算模组 | $480 \div 60$ | `hours(480, 60)` | 8.0 h |
| 第一步 加上电机 | $480 \div 360$ | `hours(480, 360)` | 1.33 h |
| 第三步 网络 p50 / p99 | 1000 次模拟 | `percentile(network_samples(1000), 50)` | 16.6 / 219.9 ms |

Thor 120 W 下 123.97 ms 来自 GR00T 2 白皮书，π0 板外多 13 ms 来自 π0 论文；
电池 480 Wh、电机平均 300 W、网络抖动和丢包的参数都是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import random

**函数 `energy_j`**：一次推理耗多少能量：能量（焦耳）= 功率（瓦）× 时间（秒）。

In [ ]:
def energy_j(watt: float, seconds: float) -> float:
    """一次推理耗多少能量：能量（焦耳）= 功率（瓦）× 时间（秒）。"""
    return watt * seconds

**函数 `hours`**：电池能撑多久：电池能量（瓦时）÷ 负载功率（瓦）。

In [ ]:
def hours(battery_wh: float, load_w: float) -> float:
    """电池能撑多久：电池能量（瓦时）÷ 负载功率（瓦）。"""
    return battery_wh / load_w

**函数 `network_samples`**：板外推理的网络时间：基础 13 ms + 随机拖尾；有 loss 的概率丢包重传，多等 retry_ms。

In [ ]:
def network_samples(n: int, seed: int = 0, base_ms: float = 13, tail_ms: float = 5, loss: float = 0.02,
                    retry_ms: float = 200) -> list[float]:
    """板外推理的网络时间：基础 13 ms + 随机拖尾；有 loss 的概率丢包重传，多等 retry_ms。"""
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        t = base_ms + rng.expovariate(1 / tail_ms)
        if rng.random() < loss:
            t += retry_ms
        out.append(t)
    return out

**函数 `percentile`**：第 p 百分位：从小到大排好，取第 p% 位置上的数。

In [ ]:
def percentile(xs: list[float], p: float) -> float:
    """第 p 百分位：从小到大排好，取第 p% 位置上的数。"""
    s = sorted(xs)
    return s[min(len(s) - 1, int(p / 100 * len(s)))]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：一次推理耗多少能量（按 120 W 功率模式上限算，偏大）")
print(f"  120 W × 0.12397 s = {energy_j(120, 0.12397):.1f} J")

In [ ]:
print("\n第一步：电池能撑多久（480 Wh、电机平均 300 W 是编的）")
print(f"  只算计算模组 60 W：{hours(480, 60):.1f} h")
print(f"  加上电机 300 W：{hours(480, 360):.2f} h，计算占总功率 {60 / 360:.1%}")

In [ ]:
print("\n第三步：板外推理的网络时间分布（参数是编的，种子 0，1000 次）")
xs = network_samples(1000)
print(f"  中位数 p50 = {percentile(xs, 50):.1f} ms，p99 = {percentile(xs, 99):.1f} ms")
print(f"  π0 板上 73 ms；板外按 p50 约 {73 + percentile(xs, 50):.0f} ms，按 p99 约 {73 + percentile(xs, 99):.0f} ms")

## 3. 代码里才看得到的

- 拖尾用的是指数分布（`expovariate`），这是最简单的选择；真实 Wi‑Fi 的分布更复杂，要用自己机器人的实测数据替换。
- `percentile` 用的是"排序后取第 p% 个"的简单写法，和 numpy 的插值写法在小样本时略有差别。

## 4. 自己改一改

- 把 `loss` 改成 0，看 p99 降到多少——丢包对拖尾的影响有多大。
- 把 `retry_ms` 改成 50（更快的重传），看 p99 怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 能量等于功率乘时间，续航要算总功率；
> 档位换速度，热了会降频；
> 上云平均快，拖尾丢包断网才要命；
> 危险紧的留身上，慢的上云带退路。